# 11.04 · Backends, routing, and permissions

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1.5–2 h |
| 📋 Prerequisites | Lesson 11.03; Lesson 03.04 (long-term memory with a store). |
| 🔑 Keys | none: every cell runs offline (no tokens) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Choose** a backend: `StateBackend` (thread), `StoreBackend` (across threads, per namespace), `FilesystemBackend` (real disk), sandboxes (Lesson 11.09).
2. **Route** path prefixes to different backends with `CompositeBackend`, and isolate data per user with a namespace factory.
3. **Protect** files with `FilesystemPermission` rules (first match wins) and know what they don't cover.
4. **Add** custom policy (logging, validation) by wrapping a backend.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26)

| Page | Read it for |
|---|---|
| [Backends](https://docs.langchain.com/oss/python/deepagents/backends) | Built-in backends, routing, namespace factories, policy hooks, the protocol |
| [Permissions](https://docs.langchain.com/oss/python/deepagents/permissions) | Rule structure, first-match-wins, ordering, subagent permissions, composite backends |
| [Going to production](https://docs.langchain.com/oss/python/deepagents/going-to-production) | Scoping files per user, assistant and thread |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-backends-permissions'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Same helpers as Lesson 11.03.
from langchain_core.messages import AIMessage

from mastery.deep import build_deep_agent
from mastery.testing import ScriptedChatModel


def call(name: str, args: dict, call_id: str = "1") -> AIMessage:
    return AIMessage("", tool_calls=[{"name": name, "args": args, "id": call_id}])


def user(text: str) -> dict:
    return {"messages": [{"role": "user", "content": text}]}


def tool_results(result) -> list[str]:
    return [m.content for m in result["messages"] if m.type == "tool"]

---
## Part 1 · One file API, many backends

The file tools never touch storage directly: they call a **backend** (`ls`, `read`, `write`, `edit`, `glob`, `grep`, optional `delete`). Swap the backend and the same tools write to state, a database or a disk.

| Backend | Lives | Shared across | Use it for |
|---|---|---|---|
| `StateBackend` (default) | graph state + checkpointer | one thread (and its subagents) | scratch, drafts, offloaded results |
| `StoreBackend(namespace=...)` | a LangGraph store | every thread with the same namespace | memories, preferences, shared knowledge |
| `FilesystemBackend(root_dir, virtual_mode=True)` | real files | everyone with disk access | local dev tools, CI, a throwaway project copy |
| Sandbox backends | an isolated machine | the sandbox's lifetime | code execution (`execute`), Lesson 11.09 |

## Part 2 · Routing and per-user memory

In [ ]:
# 🧪 StoreBackend: files that outlive the thread, isolated per user by a namespace factory.
from dataclasses import dataclass

from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore


@dataclass
class Ctx:
    user_id: str


store = InMemoryStore()
backend = CompositeBackend(
    default=StateBackend(),                                                             # scratch: this thread only
    routes={"/memories/": StoreBackend(namespace=lambda rt: (rt.context.user_id, "memories"))},   # per user, all threads
)
script = ScriptedChatModel(responses=[
    call("write_file", {"file_path": "/memories/prefs.md", "content": "Prefers short answers."}),
    call("write_file", {"file_path": "/scratch.md", "content": "temporary"}, "2"), AIMessage("saved"),
    call("ls", {"path": "/memories/"}, "3"), AIMessage("listed"),
    call("ls", {"path": "/memories/"}, "4"), AIMessage("listed")])
agent = build_deep_agent(script, backend=backend, store=store, context_schema=Ctx,
                         checkpointer=InMemorySaver(), fallbacks=False)
first = agent.invoke(user("remember this"), {"configurable": {"thread_id": "a"}}, context=Ctx("ana"))
ana_later = agent.invoke(user("what do you know?"), {"configurable": {"thread_id": "b"}}, context=Ctx("ana"))
bob = agent.invoke(user("what do you know?"), {"configurable": {"thread_id": "c"}}, context=Ctx("bob"))
print("in thread state :", list(first["files"]))
print("ana, new thread :", tool_results(ana_later)[-1])
print("bob, new thread :", tool_results(bob)[-1])
print("in the store    :", [(item.namespace, item.key) for item in store.search(("ana", "memories"))])

**What just happened?** `CompositeBackend` sent `/memories/…` to the store and everything else to state. Ana's preference came back in a **new thread**; Bob saw nothing, because the namespace factory put Ana's file under `("ana", "memories")`. The scratch file stayed in thread `a`'s state.

The namespace factory gets the LangGraph `Runtime`: `rt.context` (your context schema, here `user_id`), `rt.server_info` (assistant, authenticated user, on a LangGraph server) and `rt.execution_info` (thread and run IDs). Namespace parts may use letters, digits and `-_.@+:~`; wildcards are rejected.

> ⚠️ **Production note: never share one namespace between users.** Without a namespace factory, all users of an assistant share files. In production: namespace by the **authenticated** user (`rt.server_info.user.identity` on a LangGraph server), never by a user ID the client sends unchecked, and use a persistent store (Postgres), not `InMemoryStore`.

## Part 3 · Real disk, carefully

In [ ]:
# 🧪 FilesystemBackend on a THROWAWAY folder, behind a route, with virtual_mode=True.
import tempfile
from pathlib import Path

from deepagents.backends import FilesystemBackend

project = Path(tempfile.mkdtemp(prefix="m11-project-"))      # never your real repo or home folder
(project / "notes.md").write_text("Kickoff on Monday.", encoding="utf-8")
(project / ".env").write_text("API_KEY=not-a-real-key", encoding="utf-8")

disk_backend = CompositeBackend(default=StateBackend(),
                                routes={"/project/": FilesystemBackend(root_dir=project, virtual_mode=True)})
script = ScriptedChatModel(responses=[
    call("read_file", {"file_path": "/project/notes.md"}),
    call("read_file", {"file_path": "/project/../../etc/passwd"}, "2"),
    call("read_file", {"file_path": "/project/.env"}, "3"),
    AIMessage("done")])
unguarded = build_deep_agent(script, backend=disk_backend, fallbacks=False).invoke(user("look around"))
for text in tool_results(unguarded):
    print("tool →", text.replace("\n", " ⏎ ")[:80])

**You should now see** the note, `Path traversal not allowed` for the `../../etc/passwd` trick (that is `virtual_mode=True` at work), and **the fake API key**: nothing stopped the agent reading `.env`.

> ⚠️ **Production note: `FilesystemBackend` is for local development only.** The docs warn that it gives the agent real read/write access, including to secrets, that changes are permanent, and that `virtual_mode=False` gives **no** protection even with `root_dir`. In production: `StateBackend` / `StoreBackend` for agent files, a sandbox for code, least-privilege permissions, and logs of file operations. Never point it at your home folder or a real repo; use a throwaway copy.

Route it under a prefix (here `/project/`) instead of using it as the default: the harness writes internal files (offloaded tool results, conversation history) to the **default** backend, and you don't want those on disk.

## Part 4 · Permissions

In [ ]:
# 🧪 Permissions: first matching rule wins; no match = allowed. Specific rules FIRST.
from deepagents import FilesystemPermission

from mastery.deep import deny_everything_else, secret_deny_rules

rules = [
    *secret_deny_rules(),                                                               # .env, *.pem, *.key, secrets/
    FilesystemPermission(operations=["read", "write"], paths=["/project/**"], mode="allow"),
    deny_everything_else(["write"]),     # writes only in /project/ ("/**" + dot-file patterns, see Part 4)
]
script = ScriptedChatModel(responses=[
    call("read_file", {"file_path": "/project/.env"}),
    call("write_file", {"file_path": "/project/summary.md", "content": "Kickoff Monday."}, "2"),
    call("write_file", {"file_path": "/elsewhere.md", "content": "x"}, "3"),
    AIMessage("done")])
guarded = build_deep_agent(script, backend=disk_backend, permissions=rules, fallbacks=False).invoke(user("work"))
for text in tool_results(guarded):
    print("tool →", text[:80])
print("on disk:", sorted(p.name for p in project.iterdir()))

**You should now see** `permission denied` for reading `.env`, the summary written, the write outside `/project/` denied, and `summary.md` on disk. Rules are checked **in order**; the first whose operations and path match decides; if none matches, the call is **allowed**. `"read"` covers `ls`, `read_file`, `glob`, `grep`; `"write"` covers `write_file`, `edit_file`, `delete`. A third mode, `"interrupt"`, pauses for human approval (Lesson 11.08).

In [ ]:
# 🧪 The rule-ordering bug from the docs: a broad allow BEFORE the specific deny means the deny never fires.
from deepagents.backends.utils import create_file_data

seeded = {"/workspace/secrets/api.txt": create_file_data("API_KEY=not-a-real-key")}   # a state file this time
wrong = [FilesystemPermission(operations=["read", "write"], paths=["/workspace/**"], mode="allow"),
         *secret_deny_rules()]
right = [*secret_deny_rules(),
         FilesystemPermission(operations=["read", "write"], paths=["/workspace/**"], mode="allow")]
for name, order in [("wrong order", wrong), ("right order", right)]:
    script = ScriptedChatModel(responses=[call("read_file", {"file_path": "/workspace/secrets/api.txt"}), AIMessage("ok")])
    out = build_deep_agent(script, permissions=order, fallbacks=False).invoke({**user("read"), "files": seeded})
    print(f"{name} →", tool_results(out)[0].replace("\n", " ⏎ ")[:70])

**What just happened?** In the wrong order the broad `allow` matched first, so the secret deny never ran and the key leaked; in the right order it is denied. With `FilesystemBackend` routes, rules are also checked against the symlink-resolved target, and a deny on either blocks.

Now the gotcha. The docs' "deny all" example uses `"/**"`, but the glob matcher deepagents uses (wcmatch, without its dot-file flag) does **not** let `*` or `**` match names that start with a dot: Specific rules first, broad rules last: the course puts `secret_deny_rules()` at the top of every rule list.

In [ ]:
# 🧪 A gotcha we measured (deepagents 0.7.26): `**` does NOT match dot-files, so a "deny all" on "/**" misses /.env.
from mastery.deep import deny_everything_else

dot_seed = {"/.env": create_file_data("API_KEY=not-a-real-key")}
naive_deny_all = [FilesystemPermission(operations=["read", "write"], paths=["/**"], mode="deny")]
for name, order in [("deny '/**'          ", naive_deny_all), ("deny_everything_else", [deny_everything_else()])]:
    script = ScriptedChatModel(responses=[call("read_file", {"file_path": "/.env"}), AIMessage("ok")])
    out = build_deep_agent(script, permissions=order, fallbacks=False).invoke({**user("read"), "files": dot_seed})
    print(f"{name} →", tool_results(out)[0].replace("\n", " ⏎ ")[:70])

**What just happened?** The naive catch-all let the agent read `/.env`; `deny_everything_else()` adds `/**/.*` and `/**/.*/**` and blocks it. Dot-files are exactly where secrets live (`.env`, `.git/config`, `.aws/`), so: name secrets explicitly **first** (`secret_deny_rules()` does) and close lists with `deny_everything_else()`. Re-test this on every `deepagents` upgrade (an upgrade gate, Lesson 11.11).

**What permissions do not cover** (docs): custom tools and MCP tools that touch files, and sandbox backends, where `execute` can reach any path. With a sandbox as the default backend, every permission path must sit under a routed prefix, or `create_deep_agent` raises `NotImplementedError`. Subagents inherit the parent's rules unless their spec sets `permissions`, which **replaces** them.

## Part 5 · Policy hooks for custom logic

In [ ]:
# 🧪 A policy hook: wrap any backend to log every write (permissions can't do custom logic).
from deepagents.backends.protocol import BackendProtocol

WRITE_LOG = []


class LoggedWrites(BackendProtocol):
    """Delegates everything to `inner`; records writes and edits (an audit trail of file changes)."""

    def __init__(self, inner):
        self.inner = inner

    def __getattr__(self, name):                       # ls, read, glob, grep, delete ... unchanged
        return getattr(self.inner, name)

    def write(self, file_path, content):
        WRITE_LOG.append(("write", file_path, len(content)))
        return self.inner.write(file_path, content)

    def edit(self, file_path, old_string, new_string, replace_all=False):
        WRITE_LOG.append(("edit", file_path, len(new_string)))
        return self.inner.edit(file_path, old_string, new_string, replace_all)


logged = CompositeBackend(default=StateBackend(),
                          routes={"/project/": LoggedWrites(FilesystemBackend(root_dir=project, virtual_mode=True))})
script = ScriptedChatModel(responses=[
    call("write_file", {"file_path": "/project/todo.md", "content": "- book room"}), AIMessage("done")])
build_deep_agent(script, backend=logged, permissions=rules, fallbacks=False).invoke(user("note it"))
print("write log:", WRITE_LOG)

**What just happened?** The log shows `/todo.md`: a routed backend sees paths **relative to its route** (`/project/` stripped). Permissions are declarative paths; anything smarter (logging, size limits, content checks, rate limits) goes in a backend **wrapper**. Backends return errors in their result objects instead of raising, so a wrapper that refuses something returns `WriteResult(error=...)`.

In [ ]:
# 🧪 The course layout, used from now on: mastery.deep.course_backend + secret_deny_rules.
from mastery.deep import course_backend

reports_dir = Path(tempfile.mkdtemp(prefix="m11-reports-"))
layout = course_backend(reports=FilesystemBackend(root_dir=reports_dir, virtual_mode=True))
print({prefix: type(b).__name__ for prefix, b in layout.routes.items()}, "| default:", type(layout.default).__name__)

**What just happened?** `course_backend()` is the layout the rest of the module (and the project) uses: scratch in state, `/memories/` per user in the store (`context.user_id`), and `/reports/` wherever you route it.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| Store files missing | No `store=` on the agent | `build_deep_agent(..., store=InMemoryStore())` (a deployment provides one) |
| `AttributeError: ... user_id` in the namespace factory | No `context_schema` / `context=` | Pass both |
| A deny rule never fires | A broader allow comes first | Put specific rules first |
| `NotImplementedError` with permissions | Sandbox default backend + a path outside any route | Scope rules under routed prefixes |
| Agent files appear in your project folder | `FilesystemBackend` as the default backend | Route it under a prefix; keep `StateBackend` as default |
| Deprecation warning about backend factories | `backend=lambda rt: ...` | Pass instances (factories deprecated since 0.5.0, shims removed in 0.7) |

## 🏋️ Exercises

1. **(Easy)** Add a route `/memories/projects/` to a second `StoreBackend` namespaced `(user_id, "projects")`. Which backend handles `/memories/projects/x.md`?
2. **(Medium)** Make `/memories/` read-only for the agent with one permission rule, and show that `write_file` there is denied but `read_file` works.
3. **(Medium)** Extend `LoggedWrites` to refuse files larger than 2,000 characters with a `WriteResult(error=...)`.
4. **(Stretch)** Give a reviewer subagent its own permissions (read-only on `/project/**`, deny everything else) while the main agent can write. Sketch the `subagents=[{...}]` spec.

<details>
<summary><b>Show solutions</b></summary>

**1.** The `/memories/projects/` backend: longer prefixes win.

**2.**
```python
ro = [FilesystemPermission(operations=["write"], paths=["/memories/**"], mode="deny")]
s = ScriptedChatModel(responses=[call("write_file", {"file_path": "/memories/x.md", "content": "x"}),
                                 call("read_file", {"file_path": "/memories/prefs.md"}, "2"), AIMessage("ok")])
out = build_deep_agent(s, backend=backend, store=store, context_schema=Ctx, permissions=ro,
                       fallbacks=False).invoke(user("try"), context=Ctx("ana"))
print(tool_results(out))
```

**3.**
```python
from deepagents.backends.protocol import WriteResult

class SizeLimited(LoggedWrites):
    def write(self, file_path, content):
        if len(content) > 2000:
            return WriteResult(error=f"{file_path}: files are limited to 2,000 characters")
        return super().write(file_path, content)
```

**4.**
```python
reviewer = {"name": "reviewer", "description": "Reviews files, never edits them.",
            "system_prompt": "Review the files under /project/ and report problems.",
            "permissions": [FilesystemPermission(operations=["write"], paths=["/**"], mode="deny"),
                            FilesystemPermission(operations=["read"], paths=["/project/**"], mode="allow"),
                            FilesystemPermission(operations=["read"], paths=["/**"], mode="deny")]}
agent = build_deep_agent(backend=disk_backend, permissions=rules, subagents=[reviewer])
```
The subagent's list **replaces** the parent's, so repeat any secret rules you still need.
</details>

## 🔎 Docs drill (5 min)

Which built-in tools does a `"read"` permission cover, and which does `"write"` cover? Is a call allowed when no rule matches?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Permissions](https://docs.langchain.com/oss/python/deepagents/permissions) → Rule structure.
2. `"read"`: `ls`, `read_file`, `glob`, `grep`. `"write"`: `write_file`, `edit_file`, `delete`.
3. No match = **allowed** (permissive default), which is why the course ends lists with an explicit deny when it wants a closed default.

</details>

## ✅ Checkpoint

- [ ] I can pick a backend by lifetime and sharing, and route prefixes with `CompositeBackend`.
- [ ] My store namespaces isolate users, and I know which runtime field to use in a deployment.
- [ ] I put specific permission rules first and know what permissions don't cover.
- [ ] I only use `FilesystemBackend` on throwaway folders, with `virtual_mode=True`, behind a route.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

Check [Backends](https://docs.langchain.com/oss/python/deepagents/backends) and [Permissions](https://docs.langchain.com/oss/python/deepagents/permissions) for changes since 0.7.26: any new built-in backend, and whether interrupt-mode permissions (needs ≥ 0.6.8) or binary offloading (≥ 0.7.21) changed. Log them with versions in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Backends decide where files live; `CompositeBackend` routes prefixes; longer prefixes win.
- `StoreBackend` + a namespace factory = memory per user across threads.
- Permissions: first match wins, no match allows; secrets first; they don't cover custom tools or sandboxes.
- `FilesystemBackend` is local-dev only: throwaway folder, `virtual_mode=True`, routed, guarded.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-05-subagents-and-delegation.ipynb** → [`11-05-subagents-and-delegation.ipynb`](11-05-subagents-and-delegation.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)